## __Aprendizaje no supervisado__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

__Asunto__: Isolation Forest (iForest)

***

In [ ]:
## Librerias
from itertools import product
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from pandas import DataFrame
from numpy import array, ones, concatenate, linspace
from numpy.random import seed, randn, uniform

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

__Dataset:__

In [ ]:
seed(0)

## Numero de muestras, y de outliers
n_samples, n_outliers = 360, 100

## matriz de covarianza
covariance = array([[0.5, -0.1], [0.7, 0.4]])

## Generación de grupo de puntos
cluster_1 = 0.4 * randn(n_samples, 2) @ covariance + array([2, 2])  # general
cluster_2 = 0.3 * randn(n_samples, 2) + array([-2, -2])  # spherical
outliers = uniform(low=-4, high=4, size=(n_outliers, 2))

## Armado de dataset
X = concatenate([cluster_1, cluster_2, outliers])
y = concatenate(
    [ones((2 * n_samples), dtype=int), -ones((n_outliers), dtype=int)]
)

print('(shape) X: {} - y: {}'.format(X.shape, y.shape))

In [ ]:
## Partición de datos
X_train_Val, X_test, y_train_val, y_test = train_test_split(X, y, 
                                                            test_size=0.121, 
                                                            stratify=y, 
                                                            random_state=9001)

X_train, X_val, y_train, y_val = train_test_split(X_train_Val, y_train_val, 
                                                  test_size=0.138, 
                                                  stratify=y_train_val, 
                                                  random_state=9001)

print('(shape - Train) X: {} - y: {}'.format(X_train.shape, y_train.shape))
print('(shape - Validate) X: {} - y: {}'.format(X_val.shape, y_val.shape))
print('(shape - Test) X: {} - y: {}'.format(X_test.shape, y_test.shape))

__Visualización de los datos considerados__

In [ ]:
plt.figure(figsize=(7, 7))
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                )
plt.title("Nube de puntos")
plt.xlabel('x'), plt.ylabel('y')
plt.tight_layout()
plt.show()

## Clase IsolationForest

```{python}
    IsolationForest(n_estimators=100, 
                    max_samples='auto', 
                    contamination='auto', 
                    max_features=1.0, 
                    bootstrap=False, 
                    n_jobs=None, 
                    random_state=None, 
                    verbose=0)
```

| Parámetros | Descripción |
|------------|-------------|
| n_estimators | número de estimadores (árboles) a construir (por defecto, 100). |
| max_samples | Si es entero, representa el número de observaciones a seleccionar. Si es decimal (0,1), es porcentaje del total de muestras. Si es 'auto' se selecciona ``min(250, muestras total)`` observaciones (por defecto, 'auto'). |
| contamination | porcentaje de outliers en la data (0, 0.5], o 'auto' (por defecto, 'auto') |
| max_features | Si es entero, representa el número de features a seleccionar. Si es decimal (0,1), es porcentaje del total de features. | 
| bootstrap | Si es True, aplica muestreo con reemplazo. S es False, aplica muestreo sin reemplazo. | 
| n_jobs | número de procesos paralelos a usar (por defecto, 1) |
| random_state | (Optional): semilla de aleatoriedad | 
| verbose | 0 no mostrar salida de iteraciones, 1 mostrar detalles (por defecto, 0). | 

<br>

|Funciones | Descripción |
|----------|-------------|
| fit(X) | Entrena el modelo con los parametros asignados.|
| fit_predict(X) | Entrena el modelo e identifica si una observación es outlier (-1) o no (1). |
| predict(X) | Identifica si una observación es outlier (-1) o no (1). |



In [ ]:
## Instancia del modelo
model = IsolationForest(n_estimators=100,
                        contamination='auto',
                        random_state=9001,
                        n_jobs=-1)

## Ajuste del modelo
model.fit(X_train)

## Etiquetado de las observación si es o no outliers
etiquetado = model.predict(X_train)

## Mostrar cantidad de outliers
print('Cantidad de outliers detectado: {}'.format((etiquetado == -1).sum()))

## Mostrar las etiquetas
etiquetado[:40]


#### Graficación de los puntos con etiquetas

In [ ]:
plt.figure(figsize=(7, 7))
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                hue=etiquetado,
                palette=sns.color_palette()[:2])
plt.legend(labels=["inliers", "outliers"], title="True class")
plt.title("iForest")
plt.tight_layout()
plt.show()

#### Si disponemos de información de los outliers provenientes de expertos. 

In [ ]:
plt.figure(figsize=(14, 7))
plt.subplot(1, 2, 1)
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                hue=etiquetado,
                palette=sns.color_palette()[:2])
plt.legend(labels=["inliers", "outliers"], title="True class")
plt.title("iForest")

plt.subplot(1, 2, 2)
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                hue=y_train,
                palette=sns.color_palette()[:2]
                )
plt.legend(labels=["inliers", "outliers"], title="True class")
plt.title("Reales")
plt.xlabel('x'), plt.ylabel('y')
plt.tight_layout()
plt.show()


In [ ]:
## Calculo de rendimiento del modelo si tenemos información de los outliers reales.
NroOutlier_predicha = (etiquetado == -1).sum()
NroOutlier_verdaderos = (y_train == -1).sum()
NroOutlier_acetados = ((y_train == -1) & (etiquetado == -1)).sum()
rateOutlier_acertados = ((y_train == -1) & (etiquetado == -1)).sum() / max(NroOutlier_predicha, NroOutlier_verdaderos)

print('Cantidad de outliers detectados: {}'.format(NroOutlier_predicha))
print('Cantidad de outliers verdaderos: {}'.format(NroOutlier_verdaderos))
print('Cantidad de outliers verdaderos-detectados: {}'.format(NroOutlier_acetados))
print('Porcentaje de outliers verdaderos-detectados: {:.2f}%'.format(100*rateOutlier_acertados))

## Búsqueda de la mejor configuración

In [ ]:
## Duración: 28 min aprox en colab

## Almacenador de resultados
output = {'n_estimadores': [], 
          'contaminacion': [],
          'rate_train_aciertos': [],
          'rate_val_aciertos': []}

## Lista de hiperparametros
lista_estimadores = range(2, 200)
lista_contaminacion = linspace(0.05, 0.5, 30)

grilla = list(product(lista_estimadores, 
                 lista_contaminacion))

for n_estimadores, contaminacion in tqdm(grilla):
    
    ## Instancia del modelo
    model = IsolationForest(n_estimators=n_estimadores,
                            contamination=contaminacion,
                            random_state=9001,
                            n_jobs=-1)

    ## Ajuste del modelo
    model.fit(X_train)
    etiquetado_train = model.predict(X_train)
    etiquetado_val = model.predict(X_val)

    ## Porcentaje de puntos anómalos acertados con respecto a los verdaderos
    acertados_train = ((y_train == -1) & (etiquetado_train == -1)).sum() / max((etiquetado_train==-1).sum(),(y_train==-1).sum())
    acertados_val = ((y_val == -1) & (etiquetado_val == -1)).sum() / max((etiquetado_val==-1).sum(),(y_val==-1).sum())

    ## Almcenar los resultados
    output['n_estimadores'].append(n_estimadores)
    output['contaminacion'].append(contaminacion)
    output['rate_train_aciertos'].append(acertados_train)
    output['rate_val_aciertos'].append(acertados_val)

output = DataFrame(output).sort_values(by=['rate_val_aciertos', 'contaminacion'], ascending=False)
output.head(10)


In [ ]:
## Instancia del modelo
model = IsolationForest(n_estimators=137,
                        contamination=0.143103,
                        random_state=9007,
                        n_jobs=-1)

## Ajuste del modelo y Etiquetado de las observación si es o no outliers
model.fit(X_train_Val)
etiquetado = model.predict(X_train_Val)

## Mostrar resumen comparativo
NroOutlier_predicha = (etiquetado == -1).sum()
NroOutlier_verdaderos = (y_train_val == -1).sum()
NroOutlier_acetados = ((y_train_val == -1) & (etiquetado == -1)).sum()
rateOutlier_acertados = ((y_train_val == -1) & (etiquetado == -1)).sum() / max(NroOutlier_predicha, NroOutlier_verdaderos)

print('Cantidad de outliers detectados: {}'.format(NroOutlier_predicha))
print('Cantidad de outliers verdaderos: {}'.format(NroOutlier_verdaderos))
print('Cantidad de outliers verdaderos-detectados: {}'.format(NroOutlier_acetados))
print('Porcentaje de outliers verdaderos-detectados: {:.2f}%'.format(100*rateOutlier_acertados))

In [ ]:
## Detección de outliers en el conjunto de test
etiquetado_test = model.predict(X_test)
NroOutlier_predicha = (etiquetado_test == -1).sum()
NroOutlier_verdaderos = (y_test == -1).sum()
NroOutlier_acetados = ((y_test == -1) & (etiquetado_test == -1)).sum()
rateOutlier_acertados = ((y_test == -1) & (etiquetado_test == -1)).sum() / max(NroOutlier_predicha, NroOutlier_verdaderos)

## Mostrar resumen comparativo
print('Cantidad de outliers detectados: {}'.format(NroOutlier_predicha))
print('Cantidad de outliers verdaderos: {}'.format(NroOutlier_verdaderos))
print('Cantidad de outliers verdaderos-detectados: {}'.format(NroOutlier_acetados))
print('Porcentaje de outliers verdaderos-detectados: {:.2f}%'.format(100*rateOutlier_acertados))